In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import sys
print('nbstripouttest')

sys.path.insert(
    0,
    "/raid0/homes/jkitzmann/dev/ITK-build-binarythinning/Wrapping/Generators/Python"
)

import itk

print(itk.__file__)
print(itk.OverlapMetrics)
print(itk.CropToMaskRegion)
print(itk.SkeletonizeAirway)
print(itk.TreeLength)
print(itk.SkeletonLabeling)
print(itk.ProjectAirway)
print(itk.BranchDetection)

In [ ]:
PixelType = itk.UC
Dimension = 3
ImageType = itk.Image[PixelType, Dimension]

def _to_xyz_list(points):
    """Normalize show_points into a list of (x, y, z) tuples.

    Accepts a single ITK index (or (x, y, z) sequence), or a list of them.
    """
    if points is None:
        return []

    # a bare (x, y, z) is one point; a list/tuple of those is many
    if not (isinstance(points, (list, tuple)) and not np.isscalar(points[0])):
        points = [points]

    return [(int(p[0]), int(p[1]), int(p[2])) for p in points]

def show_mip(image, show_points=None, cmap="gray"):
    ar = itk.array_view_from_image(image)

    # ITK indices are (x, y, z); the numpy view is indexed [z, y, x]
    points = _to_xyz_list(show_points)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Axial: project along Z
    axes[0].imshow(np.max(ar, axis=0), cmap=cmap)
    axes[0].set_title("Axial MIP")
    axes[0].axis("off")

    # Coronal: project along Y
    axes[1].imshow(np.max(ar, axis=1), cmap=cmap)
    axes[1].set_title("Coronal MIP")
    axes[1].axis("off")

    # Sagittal: project along X
    axes[2].imshow(np.max(ar, axis=2), cmap=cmap)
    axes[2].set_title("Sagittal MIP")
    axes[2].axis("off")

    # each view drops the axis it projected along, so overlay the two that
    # survive it: (column, row) of x/y/z per panel
    for ax, (col, row) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
        ax.autoscale(False)
        for point in points:
            ax.scatter(
                point[col],
                point[row],
                s=80,
                facecolors="none",
                edgecolors="red",
                linewidths=1.5,
            )

    plt.tight_layout()
    plt.show()


In [ ]:
groundTruth = itk.imread(r'Sn256.nii.gz', PixelType)
prediction = itk.imread(r'Sn256.nii.gz', PixelType)
show_mip(groundTruth)
show_mip(prediction)

## Crop Images to Mask Region

In [ ]:
CropToMaskRegionType = itk.CropToMaskRegion[ImageType]
cropToMaskRegionGroundTruth = CropToMaskRegionType.New()
cropToMaskRegionGroundTruth.SetInput(groundTruth)
cropToMaskRegionGroundTruth.Update()
cropped_groundTruth = cropToMaskRegionGroundTruth.GetOutput()
show_mip(cropped_groundTruth)

cropToMaskRegionPrediction = CropToMaskRegionType.New()
cropToMaskRegionPrediction.SetInput(prediction)
cropToMaskRegionPrediction.Update()
cropped_prediction = cropToMaskRegionPrediction.GetOutput()
show_mip(cropped_prediction)




## Resample for same grid

In [ ]:
interpolator = itk.NearestNeighborInterpolateImageFunction.New(cropped_groundTruth)
transform = itk.IdentityTransform[itk.D, Dimension].New()

cropped_prediction_resampled = itk.resample_image_filter(
    cropped_prediction,
    transform=transform,
    interpolator=interpolator,
    use_reference_image=True,
    reference_image=cropped_groundTruth
)

In [ ]:
prediction = cropped_prediction_resampled
groundTruth = cropped_groundTruth

## Overlap Metrics

In [ ]:
OverlapMetrics = itk.OverlapMetrics[ImageType]

metrics = OverlapMetrics.New()

metrics.SetSourceImage(prediction)
metrics.SetTargetImage(groundTruth)

metrics.Update()

print("DSC:", metrics.GetDSC())
print("IoU:", metrics.GetIoU())
print("FPR:", metrics.GetFPR())
print("TPR:", metrics.GetTPR())

## Skeletonize

In [ ]:
SkeletonizeAirwayType = itk.SkeletonizeAirway[ImageType]
skeletonizeAirwayPrediction = SkeletonizeAirwayType.New()
skeletonizeAirwayPrediction.SetInput(prediction)
skeletonizeAirwayPrediction.Update()
prediction_skeleton = skeletonizeAirwayPrediction.GetOutput()

skeletonizeAirwayGroundTruth = SkeletonizeAirwayType.New()
skeletonizeAirwayGroundTruth.SetInput(groundTruth)
skeletonizeAirwayGroundTruth.Update()
groundTruth_skeleton = skeletonizeAirwayGroundTruth.GetOutput()


show_mip(prediction_skeleton)
show_mip(groundTruth_skeleton)

## Tree Length

In [ ]:
TreeLengthType = itk.TreeLength[ImageType]
treeLength = TreeLengthType.New()
treeLength.SetGroundTruthSkeleton(groundTruth_skeleton)
treeLength.SetPredictionMask(prediction)
treeLength.Update()

tld = treeLength.GetTreeLength()
print('Tree Length Detected: ', tld)
tree_length_comparison_image = treeLength.GetComparisonImage()
show_mip(tree_length_comparison_image)

## Skeleton Labeling

In [ ]:
# branch/generation IDs need a wider pixel type than the uchar mask
LabelPixelType = itk.SS
LabelImageType = itk.Image[LabelPixelType, Dimension]

SkeletonLabelingType = itk.SkeletonLabeling[ImageType, LabelImageType]
skeletonLabelingPrediction = SkeletonLabelingType.New()
skeletonLabelingPrediction.SetMaskImage(prediction)
skeletonLabelingPrediction.SetSkeletonImage(prediction_skeleton)
skeletonLabelingPrediction.SetSensitivityMultiplier(3.0)
skeletonLabelingPrediction.Update()

skeletonLabelingGroundTruth = SkeletonLabelingType.New()
skeletonLabelingGroundTruth.SetMaskImage(groundTruth)
skeletonLabelingGroundTruth.SetSkeletonImage(groundTruth_skeleton)
skeletonLabelingGroundTruth.SetSensitivityMultiplier(3.0)
skeletonLabelingGroundTruth.Update()

branchLabelsImagePrediction = skeletonLabelingPrediction.GetBranchLabelsImage()
generationLabelsImagePrediction = skeletonLabelingPrediction.GetGenerationLabelsImage()
distanceMapImagePrediction = skeletonLabelingPrediction.GetDistanceMapImage()
widestIndexPrediction = skeletonLabelingPrediction.GetWidestIndex()
print("widest index Prediction:", list(widestIndexPrediction))
show_mip(distanceMapImagePrediction, show_points=widestIndexPrediction)

branchLabelsImageGroundTruth = skeletonLabelingGroundTruth.GetBranchLabelsImage()
generationLabelsImageGroundTruth = skeletonLabelingGroundTruth.GetGenerationLabelsImage()
distanceMapImageGroundTruth = skeletonLabelingGroundTruth.GetDistanceMapImage()
widestIndexGroundTruth = skeletonLabelingGroundTruth.GetWidestIndex()
print("widest index GroundTruth:", list(widestIndexGroundTruth))
show_mip(distanceMapImageGroundTruth, show_points=widestIndexGroundTruth)


In [ ]:
branch_labels_prediction  = itk.array_view_from_image(branchLabelsImagePrediction)
generation_labels_prediction  = itk.array_view_from_image(generationLabelsImagePrediction)
print("Prediction branches   :", int(branch_labels_prediction.max()))
print("Prediction generations:", int(generation_labels_prediction.max()))
print("Prediction Branch labels")
show_mip(branchLabelsImagePrediction, show_points=widestIndexPrediction, cmap="nipy_spectral")
print("Prediction Generation labels")
show_mip(generationLabelsImagePrediction, show_points=widestIndexPrediction, cmap="nipy_spectral")

branch_labels_groundTruth  = itk.array_view_from_image(branchLabelsImageGroundTruth)
generation_labels_groundTruth  = itk.array_view_from_image(generationLabelsImageGroundTruth)
print("GroundTruth branches   :", int(branch_labels_groundTruth.max()))
print("GroundTruth generations:", int(generation_labels_groundTruth.max()))
print("GroundTruth Branch labels")
show_mip(branchLabelsImageGroundTruth, show_points=widestIndexGroundTruth, cmap="nipy_spectral")
print("GroundTruth Generation labels")
show_mip(generationLabelsImageGroundTruth, show_points=widestIndexGroundTruth, cmap="nipy_spectral")

## Label Projection

In [ ]:
ProjectAirwayType = itk.ProjectAirway[ImageType, LabelImageType]
projectAirwayPrediction = ProjectAirwayType.New()
projectAirwayPrediction.SetSegmentationImage(prediction)
projectAirwayPrediction.SetBranchSkeletonLabelsImage(branchLabelsImagePrediction)
projectAirwayPrediction.SetGenerationSkeletonLabelsImage(generationLabelsImagePrediction)
projectAirwayPrediction.SetNumberOfNeighbors(5)
projectAirwayPrediction.Update()
projectedBranchLabelsImagePrediction = projectAirwayPrediction.GetBranchLabelsImage()
projectedGenerationLabelsImagePrediction = projectAirwayPrediction.GetGenerationLabelsImage()
print("Prediction Projected branch labels")
show_mip(projectedBranchLabelsImagePrediction, cmap="nipy_spectral")
print("Prediction Projected generation labels")
show_mip(projectedGenerationLabelsImagePrediction, cmap="nipy_spectral")


ProjectAirwayType = itk.ProjectAirway[ImageType, LabelImageType]
projectAirwayGroundTruth = ProjectAirwayType.New()
projectAirwayGroundTruth.SetSegmentationImage(groundTruth)
projectAirwayGroundTruth.SetBranchSkeletonLabelsImage(branchLabelsImageGroundTruth)
projectAirwayGroundTruth.SetGenerationSkeletonLabelsImage(generationLabelsImageGroundTruth)
projectAirwayGroundTruth.SetNumberOfNeighbors(5)
projectAirwayGroundTruth.Update()
projectedBranchLabelsImageGroundTruth = projectAirwayGroundTruth.GetBranchLabelsImage()
projectedGenerationLabelsImageGroundTruth = projectAirwayGroundTruth.GetGenerationLabelsImage()
print("GroundTruth Projected branch labels")
show_mip(projectedBranchLabelsImageGroundTruth, cmap="nipy_spectral")
print("GroundTruth Projected generation labels")
show_mip(projectedGenerationLabelsImageGroundTruth, cmap="nipy_spectral")


## Branch Detection

In [ ]:
BranchDetectionType = itk.BranchDetection[ImageType, LabelImageType]
branchDetection = BranchDetectionType.New()
branchDetection.SetPredictionMask(prediction)
branchDetection.SetGroundTruthBranchLabels(projectedBranchLabelsImageGroundTruth)
branchDetection.SetGroundTruthGenerationLabels(projectedGenerationLabelsImageGroundTruth)
branchDetection.SetDetectionThresholdPercent(50.0)
branchDetection.Update()

print("branch detection:", round(branchDetection.GetBranchDetection(), 2), "%")
print("detected        :", branchDetection.GetBranchesDetected(), "of", branchDetection.GetTotalBranches())

# the trachea is generation 0, so the breakdown starts there rather than at 1
print(f"\n{'gen':>4} {'detected':>8} {'total':>6}  detection")
for generation in range(0, branchDetection.GetMaxGeneration() + 1):
    if not branchDetection.HasGeneration(generation):
        continue
    print(
        f"{generation:>4} {branchDetection.GetGenerationBranchesDetected(generation):>8}"
        f" {branchDetection.GetGenerationTotalBranches(generation):>6}"
        f"  {branchDetection.GetGenerationBranchDetection(generation):>6.2f}%"
    )


In [ ]:
print(f"{'id':>4} {'gen':>4} {'coverage':>9} {'detected':>9}")
for branch_id in range(1, branchDetection.GetMaxBranchId() + 1):
    if not branchDetection.HasBranch(branch_id):
        continue
    print(
        f"{branch_id:>4} {branchDetection.GetBranchGeneration(branch_id):>4}"
        f" {branchDetection.GetBranchCoverage(branch_id):>8.2f}%"
        f" {str(branchDetection.IsBranchDetected(branch_id)):>9}"
    )
